# Helm到Java的配置穿透與Kubernetes內部URL配置指南

本筆記本提供了一個全面的指南，說明如何配置Helm values將數據傳遞給Java應用程序屬性，在Kubernetes中安裝MinIO，並調整YAML配置以使用適當的內部URL。特別針對GPU服務後端(gpu-service-BE)項目的部署進行調整和最佳實踐。

## Helm值穿透到Java應用程序屬性的機制

在Kubernetes環境中，將Helm values.yaml中的配置值傳遞到Java應用程序的application.properties主要有以下幾種方式：

1. **環境變數**: Java應用程序可以讀取在容器中設置的環境變數。
2. **ConfigMaps/Secrets**: Helm可以生成包含屬性文件的ConfigMaps或Secrets，然後將其掛載到容器中。
3. **命令行參數**: Java應用程序也可以通過命令行參數接收屬性。

### 1. 通過環境變數傳遞配置

SpringBoot應用程序可以通過環境變數來覆蓋application.properties中的屬性。環境變數名稱使用以下轉換規則：

- 將屬性名稱轉為大寫
- 用下劃線(_)替換點(.)
- 用下劃線(_)替換連字符(-)

例如：`spring.datasource.url` 對應的環境變數為 `SPRING_DATASOURCE_URL`

## 當前GPU服務的Helm到Java配置穿透問題

在當前的gpu-service-BE項目中，Helm values到Java application.properties的配置穿透是通過環境變數實現的。讓我們看一下當前的實現方式和問題：

### 當前的配置方式（deployment.yaml）

在deployment.yaml中，環境變數是直接從values.yaml中獲取並設置給容器的：

In [ ]:
# 當前deployment.yaml中的環境變數配置（節選）
env:
  - name: SPRING_DATASOURCE_URL
    value: "{{ (index .Values.applicationProperties .Values.application.activeProfile).spring.datasource.url }}"
  - name: SPRING_DATASOURCE_USERNAME
    value: "{{ (index .Values.applicationProperties .Values.application.activeProfile).spring.datasource.username }}"
  - name: SPRING_DATASOURCE_PASSWORD
    valueFrom:
      secretKeyRef:
        name: {{ include "gpu-service-be.fullname" . }}-secrets
        key: db-password
  - name: MINIO_URL
    value: "{{ .Values.config.minioEndpoint }}"
  - name: MINIO_ACCESS_KEY
    valueFrom:
      secretKeyRef:
        name: {{ include "gpu-service-be.fullname" . }}-secrets
        key: minio-access-key

### 當前配置的問題

1. **硬編碼的外部IP地址**：values.yaml中包含硬編碼的IP地址，而不是使用Kubernetes內部服務名稱。
2. **環境不一致**：開發與生產環境配置不一致，增加了維護成本。
3. **敏感信息處理**：敏感信息雖然使用了Secret，但仍有改進空間。
4. **不正確的URL**：使用外部URL而不是Kubernetes內部服務URL，這可能導致不必要的網絡流量和延遲。

### 配置的最佳實踐

1. **使用Kubernetes服務名稱**：在同一命名空間內，服務可以使用其服務名稱互相訪問。
2. **環境一致性**：確保開發和生產環境具有相似的配置結構，僅有的區別應該是具體的值。
3. **安全處理敏感信息**：所有敏感信息都應存儲在Kubernetes Secrets中，且不應在values.yaml中硬編碼。
4. **正確的內部URL格式**：使用`<service-name>.<namespace>.svc.cluster.local`格式的完全限定域名。

## 解決方案：正確配置內部服務URL

以下是修正後的values.yaml配置示例，正確使用了Kubernetes內部服務名稱和URL：

In [ ]:
# 改進後的values.yaml（節選）
config:
  callbackBaseUrl: "http://gpu-service-be.gpu-service.svc.cluster.local:8080/api/v1/task/status"
  minioService: "minio"
  minioPort: 9000
  minioBucket: "notebooks"

# 應用特定配置
applicationProperties:
  dev:
    spring:
      datasource:
        url: "jdbc:postgresql://postgresql.gpu-service.svc.cluster.local:5432/gpu_service_be"
        username: "postgres"
      data:
        redis:
          host: "redis-master.gpu-service.svc.cluster.local"
          port: 6379
    minio:
      url: "http://minio.gpu-service.svc.cluster.local:9000"
      bucketName: "notebooks"
    app:
      frontendUrl: "https://jupyterhub.csie.ncu.edu.tw"
  prod:
    spring:
      datasource:
        url: "jdbc:postgresql://postgresql.gpu-service.svc.cluster.local:5432/gpu_service_be"
        username: "postgres"
      data:
        redis:
          host: "redis-master.gpu-service.svc.cluster.local"
          port: 6379
    minio:
      url: "http://minio.gpu-service.svc.cluster.local:9000"
      bucketName: "notebooks"
    app:
      frontendUrl: "https://jupyterhub.csie.ncu.edu.tw"
    security:
      saml2:
        entityId: "https://jupyterhub.csie.ncu.edu.tw/saml2/service-provider-metadata/ncu"
        acsLocation: "https://jupyterhub.csie.ncu.edu.tw/login/saml2/sso/ncu"

### 修正後的deployment.yaml環境變數配置

在deployment.yaml中，我們需要確保正確地將這些值傳遞給Java應用程序：

In [ ]:
# 改進後的deployment.yaml環境變數配置（節選）
env:
  - name: MINIO_URL
    value: "http://{{ .Values.config.minioService }}.{{ .Release.Namespace }}.svc.cluster.local:{{ .Values.config.minioPort }}"
  - name: MINIO_BUCKET_NAME
    value: "{{ .Values.config.minioBucket }}"
  - name: CALLBACK_BASE_URL
    value: "{{ .Values.config.callbackBaseUrl }}"
  - name: SPRING_PROFILES_ACTIVE
    value: "{{ .Values.application.activeProfile }}"
  - name: SPRING_DATASOURCE_URL
    value: "{{ (index .Values.applicationProperties .Values.application.activeProfile).spring.datasource.url }}"
  - name: SPRING_DATASOURCE_USERNAME
    value: "{{ (index .Values.applicationProperties .Values.application.activeProfile).spring.datasource.username }}"
  - name: SPRING_DATASOURCE_PASSWORD
    valueFrom:
      secretKeyRef:
        name: {{ include "gpu-service-be.fullname" . }}-secrets
        key: db-password
  - name: APP_FRONTEND_URL
    value: "{{ (index .Values.applicationProperties .Values.application.activeProfile).app.frontendUrl }}"

## 配置Ingress以匹配jupyterhub.csie.ncu.edu.tw

既然我們的網頁服務URL是jupyterhub.csie.ncu.edu.tw，我們需要確保Ingress配置正確：

In [ ]:
# 修正的Ingress配置
ingress:
  enabled: true
  className: "nginx"
  annotations:
    kubernetes.io/ingress.class: nginx
    nginx.ingress.kubernetes.io/ssl-redirect: "true"
    cert-manager.io/cluster-issuer: "letsencrypt-prod"
  hosts:
    - host: jupyterhub.csie.ncu.edu.tw
      paths:
        - path: /
          pathType: Prefix
          serviceName: gpu-service-be
          servicePort: 8080
  tls:
    - secretName: jupyterhub-csie-ncu-tls
      hosts:
        - jupyterhub.csie.ncu.edu.tw

## SAML2和安全配置

為了正確配置SAML2認證，我們需要確保entityId和acsLocation使用正確的公共URL：

In [ ]:
# SAML2配置
security:
  saml2:
    entityId: "https://jupyterhub.csie.ncu.edu.tw/saml2/service-provider-metadata/ncu"
    acsLocation: "https://jupyterhub.csie.ncu.edu.tw/login/saml2/sso/ncu"
    acsBinding: "POST"
    assertingParty:
      metadataUri: "https://portal.ncu.edu.tw/saml/metadata"
      entityId: "https://portal.ncu.edu.tw/saml/idp"
      singleSignOnUrl: "https://portal.ncu.edu.tw/saml/sso"
      singleSignOnBinding: "REDIRECT"

In [ ]:
# templates/deployment.yaml (excerpt)
apiVersion: apps/v1
kind: Deployment
metadata:
  name: {{ .Release.Name }}
  namespace: {{ include "gpu-service.namespace" . }}
  # ... other metadata